# Частина 1. Завантаження та аналіз даних VHI
## Завдання 1: Завантаження файлів за допомогою urllib, запобігання повторному завантаженню

In [1]:
import os
import urllib.request
from datetime import datetime

# Створюємо папку для збереження даних, якщо її немає
DOWNLOAD_DIR = "vhi_data"
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

def download_vhi_data():
    # Перевіряємо, чи в папці вже є завантажені файли (запобігання колізіям)
    existing_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.startswith("vhi_id_") and f.endswith(".csv")]
    if existing_files:
        print("Файли вже завантажено раніше. Пропускаємо скачування.")
        return

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Цикл по областях від 1 до 27 (пропускаємо 0 за умовою)
    for province_id in range(1, 28):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        filename = f"vhi_id_{province_id}_{now}.csv"
        filepath = os.path.join(DOWNLOAD_DIR, filename)
        
        try:
            print(f"Завантаження для області №{province_id}...")
            urllib.request.urlretrieve(url, filepath)
        except Exception as e:
            print(f"Помилка завантаження для області {province_id}: {e}")
            
download_vhi_data()

Завантаження для області №1...
Завантаження для області №2...
Завантаження для області №3...
Завантаження для області №4...
Завантаження для області №5...
Завантаження для області №6...
Завантаження для області №7...
Завантаження для області №8...
Завантаження для області №9...
Завантаження для області №10...
Завантаження для області №11...
Завантаження для області №12...
Завантаження для області №13...
Завантаження для області №14...
Завантаження для області №15...
Завантаження для області №16...
Завантаження для області №17...
Завантаження для області №18...
Завантаження для області №19...
Завантаження для області №20...
Завантаження для області №21...
Завантаження для області №22...
Завантаження для області №23...
Завантаження для області №24...
Завантаження для області №25...
Завантаження для області №26...
Завантаження для області №27...


## Завдання 2: Зчитування у Pandas Dataframe, очищення даних (Data Cleaning) та зміна індексів на українську абетку

In [4]:
import os
import glob
import pandas as pd

# Словник переіндексації: ключ - старий ID від NOAA, значення - (Новий ID за укр. абеткою, Назва області)
noaa_to_ua = {
    1: (22, "Черкаська"), 2: (24, "Чернігівська"), 3: (23, "Чернівецька"),
    4: (3, "Дніпропетровська"), 5: (4, "Донецька"), 6: (11, "Луганська"),
    7: (9, "Київська"), 8: (8, "Івано-Франківська"), 9: (21, "Хмельницька"),
    10: (10, "Кіровоградська"), 11: (12, "Львівська"), 12: (13, "Миколаївська"),
    13: (14, "Одеська"), 14: (15, "Полтава"), 15: (16, "Рівненська"),
    16: (17, "Сумська"), 17: (18, "Тернопільська"), 18: (19, "Харківська"),
    19: (20, "Херсонська"), 20: (7, "Запорізька"), 21: (6, "Закарпатська"),
    22: (2, "Волинська"), 23: (5, "Житомирська"), 24: (1, "Вінницька"),
    25: (25, "Крим"), 26: (26, "Севастополь"), 27: (27, "Київ місто")
}

DOWNLOAD_DIR = "vhi_data"

def load_and_clean_data():
    all_dfs = []
    # Шукаємо всі завантажені CSV файли в папці vhi_data
    files = glob.glob(os.path.join(DOWNLOAD_DIR, "vhi_id_*.csv"))
    
    for file in files:
        # Витягуємо оригінальний ID з імені файлу
        filename = os.path.basename(file)
        old_id = int(filename.split("_")[2])
        
        if old_id not in noaa_to_ua:
            continue
            
        new_id, region_name = noaa_to_ua[old_id]
        
        # ВИПРАВЛЕННЯ: додано параметр usecols=range(7)
        # Він каже зчитувати лише перші 7 стовпців і повністю ігнорувати хвостові коми, які ламали парсер
        df = pd.read_csv(
            file, 
            skiprows=1, 
            names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], 
            usecols=range(7)
        )
        
        # Очищення від залишків html-тегів (наприклад, </pre></tt>) у кінці файлу
        df = df.dropna(subset=['Year'])
        df = df[df['Year'].astype(str).str.isnumeric()]
        
        # Перетворюємо типи даних на числові
        df = df.astype({'Year': int, 'Week': int, 'VHI': float})
        
        # Залишаємо лише необхідні за завданням стовпці
        df = df[['Year', 'Week', 'VHI']]
        
        # Додаємо нові стовпчики з правильним українським індексом та назвою
        df['Area_ID'] = new_id
        df['Area_Name'] = region_name
        
        all_dfs.append(df)
        
    # Перевірка: якщо файлів немає, виводимо попередження
    if not all_dfs:
        print("Помилка: Не знайдено жодного файлу в папці 'vhi_data'. Переконайся, що спочатку запустив скрипт завантаження!")
        return pd.DataFrame()
        
    # Об'єднуємо всі області в один великий датафрейм
    main_df = pd.concat(all_dfs, ignore_index=True)
    return main_df

# Запуск функції очищення
df = load_and_clean_data()

# Виводимо результат, якщо датафрейм не порожній
if not df.empty:
    print("Дані успішно очищено та переіндексовано!")
    print(df.head())

Дані успішно очищено та переіндексовано!
   Year  Week    VHI  Area_ID       Area_Name
0  1982     2  47.04       10  Кіровоградська
1  1982     3  44.99       10  Кіровоградська
2  1982     4  41.29       10  Кіровоградська
3  1982     5  37.72       10  Кіровоградська
4  1982     6  34.91       10  Кіровоградська


## Завдання 3: Реалізація процедур вибірок (Ряд VHI для області за вказаний рік)

In [5]:
def get_vhi_by_year(dataframe, area_id, year):
    result = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['Year'] == year)]
    return result[['Week', 'VHI', 'Area_Name']]

# Приклад виклику для Вінницької області (ID=1) за 2020 рік
print("Ряд VHI для області 1 за 2020 рік:")
get_vhi_by_year(df, area_id=1, year=2020).head()

Ряд VHI для області 1 за 2020 рік:


,Week,VHI,Area_Name
35500,1,40.92,Вінницька
35501,2,43.19,Вінницька
35502,3,44.74,Вінницька
35503,4,45.29,Вінницька
35504,5,44.80,Вінницька


## Завдання 4: Ряд VHI за вказаний діапазон років для вказаних областей

In [6]:
def get_vhi_range(dataframe, area_ids, start_year, end_year):
    result = dataframe[(dataframe['Area_ID'].isin(area_ids)) & (dataframe['Year'].between(start_year, end_year))]
    return result

# Приклад для областей 1 та 2 з 2015 по 2017 роки
get_vhi_range(df, area_ids=[1, 2], start_year=2015, end_year=2017).head()

,Year,Week,VHI,Area_ID,Area_Name
30770,2015,1,47.82,2,Волинська
30771,2015,2,50.16,2,Волинська
30772,2015,3,50.53,2,Волинська
30773,2015,4,49.33,2,Волинська
30774,2015,5,46.84,2,Волинська


## Завдання 5: Пошук екстремумів (min, max), середнього та медіани

In [7]:
def get_stats(dataframe, area_id, year):
    sub_df = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['Year'] == year)]
    vhi_data = sub_df['VHI']
    
    stats = {
        'Min': vhi_data.min(),
        'Max': vhi_data.max(),
        'Mean': vhi_data.mean(),
        'Median': vhi_data.median()
    }
    return pd.DataFrame([stats], index=[sub_df['Area_Name'].iloc[0] if not sub_df.empty else area_id])

get_stats(df, area_id=1, year=2020)

,Min,Max,Mean,Median
Вінницька,34.48,64.12,45.911538,44.23
